In [ ]:
import os

def print_tree(root, prefix="", max_depth=None, depth=0):
    if max_depth is not None and depth > max_depth:
        return
    try:
        entries = sorted(os.listdir(root))
    except (PermissionError, NotADirectoryError):
        return
    dirs = [e for e in entries if os.path.isdir(os.path.join(root, e))]
    files = [e for e in entries if os.path.isfile(os.path.join(root, e))]

    # 파일이 너무 많으면 개수만 표시
    for i, d in enumerate(dirs):
        is_last = (i == len(dirs) - 1) and not files
        connector = "└── " if is_last else "├── "
        print(prefix + connector + d + "/")
        extension = "    " if is_last else "│   "
        print_tree(os.path.join(root, d), prefix + extension, max_depth, depth + 1)

    if len(files) > 10:
        print(prefix + f"└── [파일 {len(files)}개]")
    else:
        for i, f in enumerate(files):
            connector = "└── " if i == len(files) - 1 else "├── "
            print(prefix + connector + f)


ROOT = "wheelchair_datasets"   # 압축 푼 폴더 경로로 변경
print(ROOT + "/")
print_tree(ROOT, max_depth=4)

In [ ]:
import os, glob

for yaml_path in sorted(glob.glob("wheelchair_datasets/*/*/data.yaml")):
    print("=" * 60)
    print(yaml_path)
    print("-" * 60)
    with open(yaml_path, encoding="utf-8") as f:
        print(f.read())

In [ ]:
import torch
from ultralytics import YOLO

# ── 0. GPU 확인 ──────────────────────────────────────────
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = 0 if torch.cuda.is_available() else "cpu"   # 0 = 첫 번째 GPU

# ── 1. 모델 로드 ─────────────────────────────────────────
# n/s/m/l/x 중 선택. GPU 메모리 여유 있으면 yolov8s.pt 권장
model = YOLO("yolov8s.pt")          # 사전학습 가중치에서 시작

# ── 2. 학습 ─────────────────────────────────────────────
results = model.train(
    data="wheelchair_datasets/merged/data.yaml",  # ← 통합 data.yaml 경로
    epochs=100,
    imgsz=640,
    batch=16,            # GPU OOM이면 8 또는 4로 낮추기. -1 이면 자동
    device=device,       # GPU 사용
    workers=8,
    patience=20,         # 20 epoch 동안 개선 없으면 조기 종료
    optimizer="auto",
    lr0=0.01,
    cos_lr=True,
    amp=True,            # 혼합정밀 → GPU 속도/메모리 이득
    cache=False,         # RAM 충분하면 True 로 I/O 가속

    # ── 데이터 증강 (전동휠체어 vs 유모차 오분류 완화에 도움) ──
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,

    project="runs/wheelchair",
    name="yolov8s_wabs",
    exist_ok=True,
)

# ── 3. 검증 ─────────────────────────────────────────────
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50   :", metrics.box.map50)

# ── 4. 최적 가중치로 추론 테스트 ─────────────────────────
best = YOLO("runs/wheelchair/yolov8s_wabs/weights/best.pt")
# best.predict(source="테스트이미지경로.jpg", save=True, conf=0.25)

In [1]:
import os, random, shutil

SRC = "wheelchair_datasets/merged"        # 줄일 원본(병합 폴더)
DST = "wheelchair_datasets/merged_4000"   # 결과 폴더 (새로 생성)
TARGET_TOTAL = 4000                       # 목표 총 장수
SPLITS = ["train", "valid", "test"]
SEED = 42
random.seed(SEED)

# 1) 현재 split별 장수 파악
counts = {}
for sp in SPLITS:
    d = os.path.join(SRC, sp, "images")
    counts[sp] = len(os.listdir(d)) if os.path.isdir(d) else 0
total = sum(counts.values())
print("현재:", counts, "합계", total)

# 2) 비율 유지하며 줄일 비율 계산
ratio = min(1.0, TARGET_TOTAL / total)
print(f"축소 비율: {ratio:.3f}")

# 3) split별로 비율만큼 무작위 샘플링 → 복사
for sp in SPLITS:
    src_img = os.path.join(SRC, sp, "images")
    src_lbl = os.path.join(SRC, sp, "labels")
    dst_img = os.path.join(DST, sp, "images")
    dst_lbl = os.path.join(DST, sp, "labels")
    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    images = sorted(os.listdir(src_img))
    keep_n = round(len(images) * ratio)
    keep = random.sample(images, keep_n)

    for img in keep:
        shutil.copy(os.path.join(src_img, img), os.path.join(dst_img, img))
        stem = os.path.splitext(img)[0]
        lbl = stem + ".txt"
        src_l = os.path.join(src_lbl, lbl)
        if os.path.exists(src_l):
            shutil.copy(src_l, os.path.join(dst_lbl, lbl))
    print(f"{sp}: {len(images)} → {keep_n}장")

# 4) data.yaml 복사 (path만 새 폴더로 수정)
import yaml
with open(os.path.join(SRC, "data.yaml"), encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg["path"] = os.path.abspath(DST)
with open(os.path.join(DST, "data.yaml"), "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

# 5) 결과 확인
done = {sp: len(os.listdir(os.path.join(DST, sp, "images"))) for sp in SPLITS}
print("결과:", done, "합계", sum(done.values()))
print("✅ 완료 →", os.path.join(DST, "data.yaml"))

현재: {'train': 5393, 'valid': 642, 'test': 299} 합계 6334
축소 비율: 0.632
train: 5393 → 3406장
valid: 642 → 405장
test: 299 → 189장
결과: {'train': 3406, 'valid': 405, 'test': 189} 합계 4000
✅ 완료 → wheelchair_datasets/merged_4000/data.yaml


In [2]:
import os, glob, yaml
from collections import Counter

DST = "wheelchair_datasets/merged_4000"
SPLITS = ["train", "valid", "test"]

# data.yaml에서 클래스 정보 읽기
with open(os.path.join(DST, "data.yaml"), encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
names = cfg["names"]
nc = cfg["nc"] if "nc" in cfg else len(names)
print(f"클래스 수: {nc}, 클래스명: {names}\n")

total_imgs, total_lbls = 0, 0
overall_cls = Counter()

for sp in SPLITS:
    img_dir = os.path.join(DST, sp, "images")
    lbl_dir = os.path.join(DST, sp, "labels")
    if not os.path.isdir(img_dir):
        continue

    imgs = [os.path.splitext(f)[0] for f in os.listdir(img_dir)]
    lbls = [os.path.splitext(f)[0] for f in os.listdir(lbl_dir)]
    img_set, lbl_set = set(imgs), set(lbls)

    print(f"===== {sp} =====")
    print(f"이미지 {len(imgs)}장 / 라벨 {len(lbls)}개")

    # 1) 짝 안 맞는 파일
    no_label = img_set - lbl_set      # 라벨 없는 이미지(=배경, 의도적이면 OK)
    no_image = lbl_set - img_set      # 이미지 없는 라벨(문제)
    if no_label:
        print(f"  ⚠ 라벨 없는 이미지 {len(no_label)}개 (예: {list(no_label)[:3]})")
    if no_image:
        print(f"  ❌ 이미지 없는 라벨 {len(no_image)}개 (예: {list(no_image)[:3]})")

    # 2) 라벨 내용 검사
    cls_counter = Counter()
    bad_files = []
    for lbl in os.listdir(lbl_dir):
        path = os.path.join(lbl_dir, lbl)
        with open(path, encoding="utf-8") as f:
            for ln, line in enumerate(f, 1):
                parts = line.split()
                if not parts:
                    continue
                # 형식: class cx cy w h (5개)
                if len(parts) != 5:
                    bad_files.append((lbl, ln, "필드수≠5"))
                    continue
                try:
                    c = int(parts[0])
                    coords = list(map(float, parts[1:]))
                except ValueError:
                    bad_files.append((lbl, ln, "숫자변환 실패"))
                    continue
                # 클래스 인덱스 범위
                if not (0 <= c < nc):
                    bad_files.append((lbl, ln, f"클래스 {c} 범위초과"))
                # 좌표 0~1 정규화 확인
                if any(not (0.0 <= v <= 1.0) for v in coords):
                    bad_files.append((lbl, ln, "좌표 0~1 벗어남"))
                cls_counter[c] += 1

    print(f"  클래스 분포: " +
          ", ".join(f"{names[c]}({c}):{n}" for c, n in sorted(cls_counter.items())))
    if bad_files:
        print(f"  ❌ 형식 오류 {len(bad_files)}건 (예: {bad_files[:5]})")
    else:
        print("  ✅ 라벨 형식 정상")
    print()

    total_imgs += len(imgs)
    total_lbls += len(lbls)
    overall_cls.update(cls_counter)

# 전체 요약
print("===== 전체 요약 =====")
print(f"총 이미지 {total_imgs}장 / 총 라벨 {total_lbls}개")
print("전체 클래스 분포:")
for c, n in sorted(overall_cls.items()):
    print(f"  {names[c]} ({c}): {n}개")
missing = [names[c] for c in range(nc) if overall_cls[c] == 0]
if missing:
    print(f"⚠ 전혀 등장하지 않는 클래스: {missing}")

클래스 수: 7, 클래스명: ['empty_wheelchair', 'person', 'wheelchair_user', 'people_wheelchair', 'wheelchair', 'wheelchairs', 'wheelchairs_occ']

===== train =====
이미지 3406장 / 라벨 3406개
  클래스 분포: empty_wheelchair(0):1817, person(1):1873, wheelchair_user(2):1677, people_wheelchair(3):108, wheelchair(4):11, wheelchairs(5):398, wheelchairs_occ(6):22
  ✅ 라벨 형식 정상

===== valid =====
이미지 405장 / 라벨 405개
  클래스 분포: empty_wheelchair(0):175, person(1):175, wheelchair_user(2):176, people_wheelchair(3):30, wheelchair(4):2, wheelchairs(5):151, wheelchairs_occ(6):10
  ✅ 라벨 형식 정상

===== test =====
이미지 189장 / 라벨 189개
  클래스 분포: empty_wheelchair(0):91, person(1):85, wheelchair_user(2):84, people_wheelchair(3):4, wheelchair(4):1, wheelchairs(5):67, wheelchairs_occ(6):5
  ✅ 라벨 형식 정상

===== 전체 요약 =====
총 이미지 4000장 / 총 라벨 4000개
전체 클래스 분포:
  empty_wheelchair (0): 2083개
  person (1): 2133개
  wheelchair_user (2): 1937개
  people_wheelchair (3): 142개
  wheelchair (4): 14개
  wheelchairs (5): 616개
  wheelchairs_occ (6): 37개


In [ ]:
import torch
from ultralytics import YOLO

# ── GPU 확인 ─────────────────────────────────────────────
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = 0 if torch.cuda.is_available() else "cpu"   # 0 = 첫 번째 GPU

# ── 모델 로드 ────────────────────────────────────────────
model = YOLO("yolov8n.pt")          # Nano 배포용 경량 모델

# ── 학습 ────────────────────────────────────────────────
model.train(
    data="wheelchair_datasets/merged_4000/data.yaml",   # 줄인 데이터셋
    epochs=100,
    imgsz=416,           # Nano 추론 부담 ↓ (학습도 동일 해상도 권장)
    batch=16,            # GPU OOM이면 8 또는 4로. -1이면 자동
    device=device,       # ★ GPU 사용
    workers=8,
    patience=20,         # 20 epoch 개선 없으면 조기 종료
    optimizer="auto",
    lr0=0.01,
    cos_lr=True,
    amp=True,            # 혼합정밀 → GPU 속도/메모리 이득
    cache=False,         # RAM 충분하면 True로 I/O 가속

    # 증강 (전동휠체어 vs 유모차 오분류 완화)
    degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,

    project="runs/wheelchair",
    name="yolov8n_4000",
    exist_ok=True,
)

# ── 검증 ────────────────────────────────────────────────
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50   :", metrics.box.map50)
print("최적 가중치: runs/wheelchair/yolov8n_4000/weights/best.pt")

/home/jetson/PycharmProjects/wheeling/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


CUDA available: False
Ultralytics 8.4.62 🚀 Python-3.10.12 torch-2.12.0+cu130 CPU (ARMv8 Processor rev 1 (v8l))
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=wheelchair_datasets/merged_4000/data.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_4000, nbs=64, nms=False, opset=None, optimize=False, op

In [1]:
import os
import time
import torch
from ultralytics import YOLO

# ── 0. 경로 설정 ─────────────────────────────────────────
RUN_DIR   = "runs/wheelchair/yolov8n_4000"          # 중단된 학습 폴더
LAST_PT   = os.path.join(RUN_DIR, "weights", "last.pt")   # 마지막 가중치(재개용)
BEST_PT   = os.path.join(RUN_DIR, "weights", "best.pt")   # 최고 성능 가중치

# ── 1. GPU 확인 ─────────────────────────────────────────
print("CUDA available:", torch.cuda.is_available())     # GPU 사용 가능 여부
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))        # GPU 이름 출력

# ── 2. last.pt 상태 점검 ────────────────────────────────
print("\n[체크포인트 확인]")
print("last.pt 존재:", os.path.exists(LAST_PT))         # 재개에 꼭 필요한 파일
if os.path.exists(LAST_PT):
    size = os.path.getsize(LAST_PT)                     # 파일 크기(byte)
    print("last.pt 크기:", round(size / 1024 / 1024, 2), "MB")
    print("last.pt 수정시각:", time.ctime(os.path.getmtime(LAST_PT)))

# ── 3. 재개 가능 여부 판단 후 실행 ──────────────────────
if os.path.exists(LAST_PT) and os.path.getsize(LAST_PT) > 0:
    # 3-A. 정상: 중단 지점부터 이어서 학습
    print("\n✅ last.pt 발견 → 중단 지점부터 재개합니다.")
    model = YOLO(LAST_PT)               # 마지막 가중치 로드(옵티마이저·epoch 상태 포함)
    model.train(resume=True)            # ★ resume=True 만 지정(다른 인자 추가 금지)

elif os.path.exists(BEST_PT):
    # 3-B. last.pt는 깨졌지만 best.pt는 있을 때 → 처음부터 재학습(가중치만 이어받음)
    print("\n⚠ last.pt가 없거나 손상됨. best.pt로 새 학습을 시작합니다(이어재개 아님).")
    model = YOLO(BEST_PT)               # 학습된 가중치에서 출발
    model.train(
        data="wheelchair_datasets/merged_4000/data.yaml",
        epochs=100, imgsz=416, batch=16,
        device=0 if torch.cuda.is_available() else "cpu",
        workers=8, patience=20, optimizer="auto",
        lr0=0.01, cos_lr=True, amp=True,
        degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5,
        mosaic=1.0, mixup=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        project="runs/wheelchair", name="yolov8n_4000_resume", exist_ok=True,
    )

else:
    # 3-C. 둘 다 없음 → 재개 불가, 처음부터
    print("\n❌ last.pt / best.pt 모두 없음 → 처음부터 새로 학습해야 합니다.")
    model = YOLO("yolov8n.pt")          # 사전학습 가중치에서 시작
    model.train(
        data="wheelchair_datasets/merged_4000/data.yaml",
        epochs=100, imgsz=416, batch=16,
        device=0 if torch.cuda.is_available() else "cpu",
        workers=8, patience=20, optimizer="auto",
        lr0=0.01, cos_lr=True, amp=True,
        degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5,
        mosaic=1.0, mixup=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        project="runs/wheelchair", name="yolov8n_4000", exist_ok=True,
    )

# ── 4. 검증 ────────────────────────────────────────────
metrics = model.val()                              # 검증 데이터로 성능 평가
print("\nmAP50-95:", metrics.box.map)              # 종합 정확도
print("mAP50   :", metrics.box.map50)              # IoU 0.5 기준 정확도
print("최적 가중치:", os.path.join(RUN_DIR, "weights", "best.pt"))

/home/jetson/PycharmProjects/wheeling/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


CUDA available: False

[체크포인트 확인]
last.pt 존재: False

❌ last.pt / best.pt 모두 없음 → 처음부터 새로 학습해야 합니다.
New https://pypi.org/project/ultralytics/8.4.63 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.10.12 torch-2.12.0+cu130 CPU (ARMv8 Processor rev 1 (v8l))
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=wheelchair_datasets/merged_4000/data.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mas

KeyboardInterrupt: 

In [2]:
import os
import time
import torch
from ultralytics import YOLO

# ── 0. 경로 설정 ─────────────────────────────────────────
RUN_DIR = "runs/wheelchair/yolov8n_4000"                 # 기존 학습 폴더
LAST_PT = os.path.join(RUN_DIR, "weights", "last.pt")    # 마지막 가중치
BEST_PT = os.path.join(RUN_DIR, "weights", "best.pt")    # 최고 성능 가중치

# ── 1. GPU 확인 ─────────────────────────────────────────
print("CUDA available:", torch.cuda.is_available())      # GPU 사용 가능 여부
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = 0 if torch.cuda.is_available() else "cpu"

# ── 2. 이어받을 가중치 선택 ─────────────────────────────
# last.pt(정상) > best.pt > 사전학습(yolov8n.pt) 순으로 선택
if os.path.exists(LAST_PT) and os.path.getsize(LAST_PT) > 0:
    weights = LAST_PT
    print(f"\n✅ {weights} 가중치에서 이어서 학습합니다.")
    print("   수정시각:", time.ctime(os.path.getmtime(LAST_PT)))
elif os.path.exists(BEST_PT):
    weights = BEST_PT
    print(f"\n⚠ last.pt 없음 → {weights} 가중치에서 학습합니다.")
else:
    weights = "yolov8n.pt"
    print("\n❌ 기존 가중치 없음 → 사전학습 모델로 처음부터 학습합니다.")

model = YOLO(weights)                  # 선택된 가중치 로드

# ── 3. 학습 (epochs=50, patience=4 새로 적용) ───────────
model.train(
    data="wheelchair_datasets/merged_4000/data.yaml",  # 데이터셋 경로
    epochs=50,           # 전체 50 epoch 학습
    imgsz=416,           # 입력 이미지 크기 416×416
    batch=16,            # 배치 크기(OOM이면 8 또는 4로)
    device=device,       # GPU 사용
    workers=8,           # 데이터 로딩 프로세스 수
    patience=4,          # 4 epoch 동안 개선 없으면 조기 종료(얼리스탑)
    optimizer="auto",    # 옵티마이저 자동 선택
    lr0=0.01,            # 초기 학습률
    cos_lr=True,         # 학습률 코사인 감소
    amp=True,            # 혼합정밀 학습(속도/메모리 이득)

    # 증강 (전동휠체어 vs 유모차 오분류 완화)
    degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,

    project="runs/wheelchair",      # 결과 저장 상위 폴더
    name="yolov8n_4000_e50",        # 새 결과 폴더 이름(기존과 구분)
    exist_ok=True,                  # 같은 이름 폴더 덮어쓰기 허용
)

# ── 4. 검증 ────────────────────────────────────────────
metrics = model.val()                          # 검증 데이터로 성능 평가
print("\nmAP50-95:", metrics.box.map)          # 종합 정확도
print("mAP50   :", metrics.box.map50)          # IoU 0.5 기준 정확도
print("최적 가중치: runs/wheelchair/yolov8n_4000_e50/weights/best.pt")

CUDA available: False

❌ 기존 가중치 없음 → 사전학습 모델로 처음부터 학습합니다.
New https://pypi.org/project/ultralytics/8.4.63 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.10.12 torch-2.12.0+cu130 CPU (ARMv8 Processor rev 1 (v8l))
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=wheelchair_datasets/merged_4000/data.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=tr

In [6]:
import os
import glob
import time
import cv2
import torch
from ultralytics import YOLO

# ── 0. 설정 ─────────────────────────────────────────────
CAM_ID = 1           # 웹캠 번호 (안 잡히면 1, 2로 변경)
IMGSZ  = 416         # 추론 입력 크기 (학습과 동일)
CONF   = 0.25        # 신뢰도 임계값 (이 값 이상만 표시)

# ── 1. 학습된 가중치 자동 탐색 ──────────────────────────
print("[가중치 탐색]")
print("현재 작업 경로:", os.getcwd())

pt_files = glob.glob("runs/**/weights/best.pt", recursive=True)   # 모든 best.pt 검색
if not pt_files:                                                   # best.pt 없으면 last.pt도 검색
    pt_files = glob.glob("runs/**/weights/last.pt", recursive=True)

if not pt_files:
    raise FileNotFoundError(
        "runs 폴더에서 가중치(.pt)를 찾지 못했습니다.\n"
        "  → 학습이 완료됐는지, 작업 폴더가 프로젝트 루트인지 확인하세요."
    )

# 가장 최근에 수정된 가중치 선택
WEIGHTS = max(pt_files, key=os.path.getmtime)
print("사용할 가중치:", WEIGHTS)
print("  수정시각:", time.ctime(os.path.getmtime(WEIGHTS)))
print("  크기:", round(os.path.getsize(WEIGHTS) / 1024 / 1024, 1), "MB")

# ── 2. GPU 확인 ─────────────────────────────────────────
print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = 0 if torch.cuda.is_available() else "cpu"

# ── 3. 모델 로드 ────────────────────────────────────────
model = YOLO(WEIGHTS)                 # 학습된 가중치 불러오기
print("클래스:", model.names)          # 모델이 인식하는 클래스 확인

# ── 4. 웹캠 열기 ────────────────────────────────────────
cap = cv2.VideoCapture(CAM_ID)        # 웹캠 연결
if not cap.isOpened():
    raise RuntimeError(f"웹캠({CAM_ID})을 열 수 없습니다. 카메라 번호를 확인하세요.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)    # 캡처 해상도 가로
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)   # 캡처 해상도 세로

print("\n실시간 탐지 시작 — 종료하려면 영상 창에서 'q' 키를 누르세요.\n")

# ── 5. 실시간 추론 루프 ─────────────────────────────────
while True:
    ret, frame = cap.read()           # 웹캠에서 한 프레임 읽기
    if not ret:                       # 읽기 실패 시
        print("프레임 읽기 실패")
        break

    results = model.predict(          # 추론 실행
        source=frame,                 # 현재 프레임
        imgsz=IMGSZ,                  # 입력 크기
        conf=CONF,                    # 신뢰도 임계값
        device=device,               # GPU/CPU
        verbose=False,               # 콘솔 로그 끄기
    )

    annotated = results[0].plot()     # 결과(박스+라벨)를 프레임에 그림

    boxes = results[0].boxes          # 감지된 객체들
    for c, conf in zip(boxes.cls, boxes.conf):
        name = model.names[int(c)]
        print(f"감지: {name} ({conf:.2f})")   # 감지 객체 콘솔 출력

    cv2.imshow("WABS - Wheelchair Detection", annotated)  # 화면 표시

    if cv2.waitKey(1) & 0xFF == ord("q"):   # 'q' 키 누르면 종료
        break

# ── 6. 정리 ────────────────────────────────────────────
cap.release()                         # 웹캠 자원 해제
cv2.destroyAllWindows()               # 모든 창 닫기
print("\n종료되었습니다.")

[가중치 탐색]
현재 작업 경로: /home/jetson/PycharmProjects/wheeling
사용할 가중치: runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.pt
  수정시각: Wed Jun 10 20:05:46 2026
  크기: 5.9 MB

CUDA available: False
클래스: {0: 'empty_wheelchair', 1: 'person', 2: 'wheelchair_user', 3: 'people_wheelchair', 4: 'wheelchair', 5: 'wheelchairs', 6: 'wheelchairs_occ'}


[ WARN:0@86236.186] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index



실시간 탐지 시작 — 종료하려면 영상 창에서 'q' 키를 누르세요.



QFontDatabase: Cannot find font directory /home/jetson/PycharmProjects/wheeling/.venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/jetson/PycharmProjects/wheeling/.venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/jetson/PycharmProjects/wheeling/.venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/jetson/PycharmProjects/wheeling/.venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to 

감지: empty_wheelchair (0.29)
감지: wheelchairs (0.27)
감지: wheelchairs (0.34)
감지: wheelchairs (0.40)
감지: empty_wheelchair (0.45)
감지: empty_wheelchair (0.29)
감지: empty_wheelchair (0.26)
감지: empty_wheelchair (0.39)
감지: empty_wheelchair (0.37)
감지: empty_wheelchair (0.28)
감지: empty_wheelchair (0.29)
감지: empty_wheelchair (0.51)
감지: wheelchairs (0.30)
감지: wheelchairs (0.28)
감지: wheelchairs (0.45)
감지: wheelchairs (0.67)
감지: wheelchairs (0.49)
감지: wheelchairs (0.35)
감지: wheelchairs (0.28)
감지: wheelchairs (0.53)
감지: wheelchairs (0.36)
감지: empty_wheelchair (0.56)
감지: wheelchairs (0.28)
감지: empty_wheelchair (0.31)
감지: wheelchairs (0.59)
감지: wheelchairs (0.25)
감지: wheelchairs (0.27)
감지: wheelchairs (0.39)
감지: wheelchairs (0.26)
감지: wheelchairs (0.41)
감지: wheelchairs (0.29)
감지: people_wheelchair (0.28)
감지: wheelchairs (0.33)
감지: wheelchairs (0.28)
감지: wheelchairs (0.28)
감지: wheelchairs (0.34)
감지: wheelchairs (0.27)
감지: wheelchairs (0.35)
감지: wheelchairs (0.26)
감지: wheelchairs (0.26)
감지: wheelchairs (0.

ioctl(VIDIOC_QBUF): 파일 디스크립터가 잘못됨


In [8]:
import os
import glob
import time
import cv2
import torch
from ultralytics import YOLO

# ── 0. 설정 ─────────────────────────────────────────────
CAM_ID = 1           # 웹캠 번호 (안 잡히면 0, 2로 변경)
IMGSZ  = 416         # 추론 입력 크기 (학습과 동일)
CONF   = 0.50
# ★ 신뢰도 임계값 낮춤 (0.25→0.10, 약한 탐지도 확인)

# ── 1. 학습된 가중치 자동 탐색 ──────────────────────────
print("[가중치 탐색]")
print("현재 작업 경로:", os.getcwd())

pt_files = glob.glob("runs/**/weights/best.pt", recursive=True)
if not pt_files:
    pt_files = glob.glob("runs/**/weights/last.pt", recursive=True)
if not pt_files:
    raise FileNotFoundError(
        "runs 폴더에서 가중치(.pt)를 찾지 못했습니다.\n"
        "  → 학습 완료 여부, 작업 폴더가 프로젝트 루트인지 확인하세요."
    )

WEIGHTS = max(pt_files, key=os.path.getmtime)   # 가장 최근 가중치
print("사용할 가중치:", WEIGHTS)
print("  수정시각:", time.ctime(os.path.getmtime(WEIGHTS)))
print("  크기:", round(os.path.getsize(WEIGHTS) / 1024 / 1024, 1), "MB")

# ── 2. GPU 확인 ─────────────────────────────────────────
print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = 0 if torch.cuda.is_available() else "cpu"

# ── 3. 모델 로드 + 클래스 확인 ─────────────────────────
model = YOLO(WEIGHTS)
print("\n" + "=" * 50)
print("★ 이 모델이 학습한 클래스 목록:")
for idx, name in model.names.items():
    print(f"   {idx}: {name}")
print("=" * 50)
print("→ 위 클래스가 의도한 6개(또는 3개)와 맞는지 꼭 확인하세요.")
print("→ 엉뚱한 클래스(person 등 COCO 기본)면 잘못된 가중치입니다.\n")

# ── 4. 웹캠 열기 ────────────────────────────────────────
cap = cv2.VideoCapture(CAM_ID)
if not cap.isOpened():
    raise RuntimeError(f"웹캠({CAM_ID})을 열 수 없습니다. 카메라 번호(0/1/2)를 바꿔보세요.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("실시간 탐지 시작 — 'q' 종료 / '+' conf↑ / '-' conf↓\n")

conf = CONF                          # 실행 중 조절 가능한 임계값
frame_count = 0

# ── 5. 실시간 추론 루프 ─────────────────────────────────
while True:
    ret, frame = cap.read()
    if not ret:
        print("프레임 읽기 실패")
        break

    results = model.predict(
        source=frame,
        imgsz=IMGSZ,
        conf=conf,
        device=device,
        verbose=False,
    )

    annotated = results[0].plot()    # 박스+라벨 그림
    boxes = results[0].boxes
    n = len(boxes)

    # 화면 좌상단에 디버그 정보 표시
    cv2.putText(annotated, f"conf>={conf:.2f}  detections={n}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # 감지 객체 콘솔 출력 (10프레임마다, 로그 과다 방지)
    frame_count += 1
    if n > 0 and frame_count % 10 == 0:
        for c, cf in zip(boxes.cls, boxes.conf):
            print(f"감지: {model.names[int(c)]} ({cf:.2f})")

    cv2.imshow("WABS - Wheelchair Detection", annotated)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):              # 종료
        break
    elif key == ord("+") or key == ord("="):   # 임계값 올리기
        conf = min(0.95, conf + 0.05)
        print(f"conf → {conf:.2f}")
    elif key == ord("-"):            # 임계값 내리기
        conf = max(0.05, conf - 0.05)
        print(f"conf → {conf:.2f}")

# ── 6. 정리 ────────────────────────────────────────────
cap.release()
cv2.destroyAllWindows()
print("\n종료되었습니다.")

[가중치 탐색]
현재 작업 경로: /home/jetson/PycharmProjects/wheeling
사용할 가중치: runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.pt
  수정시각: Wed Jun 10 20:05:46 2026
  크기: 5.9 MB

CUDA available: False

★ 이 모델이 학습한 클래스 목록:
   0: empty_wheelchair
   1: person
   2: wheelchair_user
   3: people_wheelchair
   4: wheelchair
   5: wheelchairs
   6: wheelchairs_occ
→ 위 클래스가 의도한 6개(또는 3개)와 맞는지 꼭 확인하세요.
→ 엉뚱한 클래스(person 등 COCO 기본)면 잘못된 가중치입니다.



[ WARN:0@88471.652] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index


실시간 탐지 시작 — 'q' 종료 / '+' conf↑ / '-' conf↓

감지: wheelchairs (0.52)
감지: wheelchairs (0.63)
감지: wheelchairs (0.75)
감지: people_wheelchair (0.53)
감지: people_wheelchair (0.53)
감지: people_wheelchair (0.51)
감지: people_wheelchair (0.51)
감지: wheelchairs (0.69)
감지: wheelchairs (0.53)
감지: wheelchairs (0.84)
감지: wheelchairs (0.76)
감지: wheelchairs (0.72)
감지: wheelchairs (0.76)
감지: wheelchairs (0.79)
감지: wheelchairs (0.82)
감지: wheelchairs (0.81)
감지: wheelchairs (0.55)

종료되었습니다.


ioctl(VIDIOC_QBUF): 파일 디스크립터가 잘못됨


In [9]:
import os
import glob
import time
import cv2
import torch
from ultralytics import YOLO

# ══════════════════════════════════════════════════════════
# 0. 설정 (속도 최적화)
# ══════════════════════════════════════════════════════════
CAM_ID = 1
IMGSZ  = 320          # ★ 416→320 축소: 추론량이 크게 줄어 속도 ↑ (정확도 약간↓)
CONF   = 0.50        # 디버깅 끝났으면 0.25로 (낮을수록 후처리 박스 많아져 느려짐)
SKIP   = 1            # ★ N프레임마다 1번만 추론 (1=매프레임, 2=격프레임). 끊기면 2로

# ══════════════════════════════════════════════════════════
# 1. 가중치 자동 탐색
# ══════════════════════════════════════════════════════════
pt_files = glob.glob("runs/**/weights/best.pt", recursive=True) \
        or glob.glob("runs/**/weights/last.pt", recursive=True)
if not pt_files:
    raise FileNotFoundError("runs 폴더에서 가중치(.pt)를 찾지 못했습니다.")
WEIGHTS = max(pt_files, key=os.path.getmtime)
print("사용할 가중치:", WEIGHTS)

# ══════════════════════════════════════════════════════════
# 2. GPU 확인  ← ★ 속도의 90%가 여기서 갈린다
# ══════════════════════════════════════════════════════════
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠⚠ GPU 미사용! CPU 추론이라 느린 게 당연합니다.")
    print("   → 이게 느림의 핵심 원인. GPU용 torch 재설치가 가장 큰 개선입니다.")
device = 0 if torch.cuda.is_available() else "cpu"

# ══════════════════════════════════════════════════════════
# 3. 모델 로드 + 반정밀(FP16)로 속도 ↑
# ══════════════════════════════════════════════════════════
model = YOLO(WEIGHTS)
use_half = torch.cuda.is_available()   # ★ GPU일 때만 FP16 가능 (CPU는 미지원)

# ══════════════════════════════════════════════════════════
# 4. 웹캠 열기 (Windows는 CAP_DSHOW가 빠르게 열림)
# ══════════════════════════════════════════════════════════
cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(CAM_ID)     # DSHOW 실패 시 기본 백엔드로 재시도
if not cap.isOpened():
    raise RuntimeError(f"웹캠({CAM_ID})을 열 수 없습니다.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)    # ★ 버퍼 1프레임: 밀린 프레임 누적(지연) 방지

print("\n실시간 탐지 시작 — 'q' 종료\n")

frame_count = 0
annotated = None
t_prev = time.time()

# ══════════════════════════════════════════════════════════
# 5. 추론 루프 (프레임 스킵 + FPS 표시)
# ══════════════════════════════════════════════════════════
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # ★ SKIP 프레임마다 한 번만 추론 → 나머지 프레임은 직전 결과 재사용
    if frame_count % SKIP == 0:
        results = model.predict(
            source=frame,
            imgsz=IMGSZ,
            conf=CONF,
            device=device,
            half=use_half,     # ★ GPU FP16 추론 (속도/메모리 이득)
            verbose=False,
        )
        annotated = results[0].plot()

    disp = annotated if annotated is not None else frame

    # FPS 계산해서 화면에 표시
    t_now = time.time()
    fps = 1.0 / max(t_now - t_prev, 1e-6)
    t_prev = t_now
    cv2.putText(disp, f"FPS: {fps:.1f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow("WABS - Wheelchair Detection", disp)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

사용할 가중치: runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.pt
CUDA available: False
⚠⚠ GPU 미사용! CPU 추론이라 느린 게 당연합니다.
   → 이게 느림의 핵심 원인. GPU용 torch 재설치가 가장 큰 개선입니다.


[ WARN:0@90153.213] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
[ WARN:0@90153.220] global cap.cpp:438 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(4.13.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1220: error: (-2:Unspecified error) in function 'bool CvCapture_FFMPEG::open(const char*, int, const cv::Ptr<cv::IStreamReader>&, const cv::VideoCaptureParameters&)'
> VIDEOIO/FFMPEG: Camera index out of range (expected: 'index < device_list->nb_devices'), where
>     'index' is 1
> must be less than
>     'device_list->nb_devices' is 0


[ERROR:0@90153.220] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


RuntimeError: 웹캠(1)을 열 수 없습니다.

In [10]:
import os
import glob
import time
import subprocess
import cv2
from ultralytics import YOLO

# ══════════════════════════════════════════════════════════
# 0. 설정 (Jetson Nano 최적화)
# ══════════════════════════════════════════════════════════
CAM_ID = 0          # Nano USB캠은 보통 0 (안 잡히면 1). CSI캠이면 아래 ★CSI 참고
IMGSZ  = 320        # ★ Nano 권장 해상도 (416보다 훨씬 빠름). 엔진과 추론 동일해야 함
CONF   = 0.25       # 신뢰도 임계값
SKIP   = 2          # ★ 격프레임 추론 (체감 끊김 완화). 더 부드럽게는 1, 더 빠르게는 3

# ══════════════════════════════════════════════════════════
# 1. Nano 최대 성능 모드 설정 (선택, 권장)
#    ─ 실패해도 무시하고 진행 (sudo 권한 없거나 PC면 그냥 넘어감)
# ══════════════════════════════════════════════════════════
for cmd in (["sudo", "nvpmodel", "-m", "0"], ["sudo", "jetson_clocks"]):
    try:
        subprocess.run(cmd, check=True, timeout=10,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("성능 모드 적용:", " ".join(cmd))
    except Exception:
        pass   # Nano가 아니거나 권한 없으면 조용히 건너뜀

# ══════════════════════════════════════════════════════════
# 2. 가중치 탐색 → 엔진 없으면 자동 변환
# ══════════════════════════════════════════════════════════
eng_files = glob.glob("runs/**/weights/*.engine", recursive=True)
pt_files  = glob.glob("runs/**/weights/best.pt", recursive=True) \
         or glob.glob("runs/**/weights/last.pt", recursive=True)

if eng_files:
    # 이미 변환된 엔진이 있으면 그대로 사용 (가장 빠름)
    WEIGHTS = max(eng_files, key=os.path.getmtime)
    print("✅ TensorRT 엔진 사용:", WEIGHTS)

elif pt_files:
    # 엔진이 없으면 .pt → .engine 변환 (최초 1회, 몇 분 소요)
    src_pt = max(pt_files, key=os.path.getmtime)
    print("⚙ 엔진 없음 → TensorRT 변환 시작 (최초 1회, 수 분 소요):", src_pt)
    YOLO(src_pt).export(
        format="engine",   # TensorRT 엔진 생성
        imgsz=IMGSZ,       # 추론과 동일 해상도
        half=True,         # FP16 (Nano 속도/메모리 이득)
        device=0,          # Nano GPU
    )
    # 변환 결과(.engine)는 .pt와 같은 폴더에 생성됨
    WEIGHTS = os.path.splitext(src_pt)[0] + ".engine"
    print("✅ 변환 완료:", WEIGHTS)

else:
    raise FileNotFoundError(
        "runs 폴더에서 가중치(.pt/.engine)를 찾지 못했습니다.\n"
        "  → 학습 완료 여부와 작업 폴더(프로젝트 루트)를 확인하세요."
    )

# ══════════════════════════════════════════════════════════
# 3. 모델 로드 + 클래스 확인
# ══════════════════════════════════════════════════════════
model = YOLO(WEIGHTS)
print("\n" + "=" * 50)
print("★ 모델이 인식하는 클래스:")
for idx, name in model.names.items():
    print(f"   {idx}: {name}")
print("=" * 50 + "\n")

# ══════════════════════════════════════════════════════════
# 4. 웹캠 열기
# ══════════════════════════════════════════════════════════
cap = cv2.VideoCapture(CAM_ID)
#   ★CSI: 라즈베리파이/CSI 카메라면 위 줄 대신 아래 GStreamer 사용
#   gst = ("nvarguscamerasrc ! video/x-raw(memory:NVMM),width=640,height=480,"
#          "framerate=30/1 ! nvvidconv ! video/x-raw,format=BGRx ! "
#          "videoconvert ! video/x-raw,format=BGR ! appsink")
#   cap = cv2.VideoCapture(gst, cv2.CAP_GSTREAMER)

if not cap.isOpened():
    raise RuntimeError(f"웹캠({CAM_ID})을 열 수 없습니다. USB캠 번호(0/1)를 확인하세요.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)     # 캡처 해상도 가로
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)    # 캡처 해상도 세로
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)        # ★ 버퍼 1: 밀린 프레임 누적(지연) 방지

print("실시간 탐지 시작 — 'q' 종료\n")

frame_count = 0
annotated = None
t_prev = time.time()

# ══════════════════════════════════════════════════════════
# 5. 추론 루프 (프레임 스킵 + FPS 표시)
# ══════════════════════════════════════════════════════════
while True:
    ret, frame = cap.read()           # 웹캠 한 프레임 읽기
    if not ret:
        print("프레임 읽기 실패")
        break

    frame_count += 1

    # ★ SKIP 프레임마다 한 번만 추론, 나머지는 직전 결과 재사용 → 부드러움 유지
    if frame_count % SKIP == 0:
        results = model.predict(
            source=frame,
            imgsz=IMGSZ,
            conf=CONF,
            half=True,        # FP16 추론 (엔진과 일관)
            verbose=False,
        )
        annotated = results[0].plot()    # 박스+라벨 그린 이미지

        # 감지 객체 콘솔 출력
        for c, cf in zip(results[0].boxes.cls, results[0].boxes.conf):
            print(f"감지: {model.names[int(c)]} ({cf:.2f})")

    disp = annotated if annotated is not None else frame

    # FPS 계산 후 화면 표시
    t_now = time.time()
    fps = 1.0 / max(t_now - t_prev, 1e-6)
    t_prev = t_now
    cv2.putText(disp, f"FPS: {fps:.1f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow("WABS - Wheelchair Detection", disp)
    if cv2.waitKey(1) & 0xFF == ord("q"):   # 'q' 종료
        break

# ══════════════════════════════════════════════════════════
# 6. 정리
# ══════════════════════════════════════════════════════════
cap.release()
cv2.destroyAllWindows()
print("\n종료되었습니다.")

⚙ 엔진 없음 → TensorRT 변환 시작 (최초 1회, 수 분 소요): runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.pt
Ultralytics 8.4.62 🚀 Python-3.10.12 torch-2.12.0+cu130 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 1
os.environ['CUDA_VISIBLE_DEVICES']: 


In [11]:
import os
import glob
import time
import subprocess
import cv2
import torch
from ultralytics import YOLO

# ══════════════════════════════════════════════════════════
# 0. 설정
# ══════════════════════════════════════════════════════════
CAM_ID = 0          # USB캠 보통 0 (안 잡히면 1)
IMGSZ  = 320        # Nano 권장 해상도
CONF   = 0.25
SKIP   = 2          # 격프레임 추론

# ══════════════════════════════════════════════════════════
# 1. 환경 진단 — GPU torch가 제대로 깔렸는지 먼저 확인
# ══════════════════════════════════════════════════════════
print("=" * 55)
print("[환경 진단]")
print("torch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())   # ★ 이게 핵심
print("CUDA 장치 수:", torch.cuda.device_count())

GPU_OK = torch.cuda.is_available()

# JetPack/L4T 버전 출력 (재설치 시 어떤 whl을 받아야 할지 판단용)
try:
    with open("/etc/nv_tegra_release") as f:
        print("JetPack/L4T:", f.readline().strip())
except FileNotFoundError:
    print("JetPack/L4T: (확인 불가 — Nano가 아니거나 파일 없음)")

if not GPU_OK:
    print("\n⚠ GPU torch가 아닙니다(CPU 전용 torch 설치됨).")
    print("  → 근본 해결: 위 JetPack 버전에 맞는 Jetson용 torch 재설치 필요.")
    print("  → 지금은 CPU(ONNX)로 자동 전환해 동작은 시킵니다(속도 제한적).")
print("=" * 55 + "\n")

# 장치/정밀도 자동 결정
DEVICE = 0 if GPU_OK else "cpu"
USE_HALF = GPU_OK              # FP16은 GPU에서만
EXPORT_FORMAT = "engine" if GPU_OK else "onnx"   # GPU=TensorRT, CPU=ONNX

# ══════════════════════════════════════════════════════════
# 2. Nano 성능 모드 (GPU 정상일 때만 의미 있음, 실패해도 무시)
# ══════════════════════════════════════════════════════════
if GPU_OK:
    for cmd in (["sudo", "nvpmodel", "-m", "0"], ["sudo", "jetson_clocks"]):
        try:
            subprocess.run(cmd, check=True, timeout=10,
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("성능 모드 적용:", " ".join(cmd))
        except Exception:
            pass

# ══════════════════════════════════════════════════════════
# 3. 가속 가중치 탐색 → 없으면 자동 변환 (GPU=engine / CPU=onnx)
# ══════════════════════════════════════════════════════════
ext = "*.engine" if GPU_OK else "*.onnx"
accel = glob.glob(f"runs/**/weights/{ext}", recursive=True)
pt_files = glob.glob("runs/**/weights/best.pt", recursive=True) \
        or glob.glob("runs/**/weights/last.pt", recursive=True)

if accel:
    WEIGHTS = max(accel, key=os.path.getmtime)
    print(f"✅ 가속 모델 사용: {WEIGHTS}")
elif pt_files:
    src_pt = max(pt_files, key=os.path.getmtime)
    print(f"⚙ 가속 모델 없음 → {EXPORT_FORMAT} 변환 시작: {src_pt}")
    YOLO(src_pt).export(
        format=EXPORT_FORMAT,     # GPU면 engine, CPU면 onnx
        imgsz=IMGSZ,
        half=USE_HALF,            # GPU만 FP16
        # device 인자는 일부러 생략 → CPU에서도 에러 없이 동작
        **({"device": 0} if GPU_OK else {}),
    )
    new_ext = ".engine" if GPU_OK else ".onnx"
    WEIGHTS = os.path.splitext(src_pt)[0] + new_ext
    print("✅ 변환 완료:", WEIGHTS)
else:
    raise FileNotFoundError("runs 폴더에서 가중치(.pt)를 찾지 못했습니다.")

# ══════════════════════════════════════════════════════════
# 4. 모델 로드 + 클래스 확인
# ══════════════════════════════════════════════════════════
model = YOLO(WEIGHTS)
print("\n★ 모델 클래스:")
for idx, name in model.names.items():
    print(f"   {idx}: {name}")
print()

# ══════════════════════════════════════════════════════════
# 5. 웹캠 열기
# ══════════════════════════════════════════════════════════
cap = cv2.VideoCapture(CAM_ID)
#   ★CSI 카메라면 위 줄 대신:
#   gst = ("nvarguscamerasrc ! video/x-raw(memory:NVMM),width=640,height=480,"
#          "framerate=30/1 ! nvvidconv ! video/x-raw,format=BGRx ! "
#          "videoconvert ! video/x-raw,format=BGR ! appsink")
#   cap = cv2.VideoCapture(gst, cv2.CAP_GSTREAMER)
if not cap.isOpened():
    raise RuntimeError(f"웹캠({CAM_ID})을 열 수 없습니다. 번호(0/1) 확인.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)     # 밀린 프레임 누적 방지

print("실시간 탐지 시작 — 'q' 종료\n")
frame_count = 0
annotated = None
t_prev = time.time()

# ══════════════════════════════════════════════════════════
# 6. 추론 루프
# ══════════════════════════════════════════════════════════
while True:
    ret, frame = cap.read()
    if not ret:
        print("프레임 읽기 실패")
        break
    frame_count += 1

    if frame_count % SKIP == 0:
        results = model.predict(
            source=frame,
            imgsz=IMGSZ,
            conf=CONF,
            device=DEVICE,        # GPU(0) 또는 cpu
            half=USE_HALF,        # GPU만 FP16
            verbose=False,
        )
        annotated = results[0].plot()
        for c, cf in zip(results[0].boxes.cls, results[0].boxes.conf):
            print(f"감지: {model.names[int(c)]} ({cf:.2f})")

    disp = annotated if annotated is not None else frame
    t_now = time.time()
    fps = 1.0 / max(t_now - t_prev, 1e-6)
    t_prev = t_now
    cv2.putText(disp, f"FPS: {fps:.1f}  {'GPU' if GPU_OK else 'CPU'}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow("WABS - Wheelchair Detection", disp)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
print("\n종료되었습니다.")

[환경 진단]
torch 버전: 2.12.0+cu130
CUDA 사용 가능: False
CUDA 장치 수: 1
JetPack/L4T: # R36 (release), REVISION: 4.7, GCID: 42132812, BOARD: generic, EABI: aarch64, DATE: Thu Sep 18 22:54:44 UTC 2025

⚠ GPU torch가 아닙니다(CPU 전용 torch 설치됨).
  → 근본 해결: 위 JetPack 버전에 맞는 Jetson용 torch 재설치 필요.
  → 지금은 CPU(ONNX)로 자동 전환해 동작은 시킵니다(속도 제한적).

⚙ 가속 모델 없음 → onnx 변환 시작: runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.pt
Ultralytics 8.4.62 🚀 Python-3.10.12 torch-2.12.0+cu130 CPU (ARMv8 Processor rev 1 (v8l))
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 11, 2100) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/17.5 MB ? eta -:--:--
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/17

2026-06-11 10:35:14.239145600 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card1/device/vendor"


ONNX: export success ✅ 15.7s, saved as 'runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.onnx' (11.6 MB)

Export complete (17.8s)
Results saved to /home/jetson/PycharmProjects/wheeling/runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.onnx
Predict:         yolo predict task=detect model=runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.onnx imgsz=320 
Validate:        yolo val task=detect model=runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.onnx imgsz=320 data=wheelchair_datasets/merged_4000/data.yaml  
Visualize:       https://netron.app
✅ 변환 완료: runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.onnx

★ 모델 클래스:
Loading runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 with CPUExecutionProvider
   0: empty_wheelchair
   1: person
   2: wheelchair_user
   3: people_wheelchair
   4: wheelchair
   5: wheelchairs
   6: wheelchairs_occ



[ WARN:0@90712.662] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@90712.663] global cap.cpp:438 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(4.13.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1220: error: (-2:Unspecified error) in function 'bool CvCapture_FFMPEG::open(const char*, int, const cv::Ptr<cv::IStreamReader>&, const cv::VideoCaptureParameters&)'
> VIDEOIO/FFMPEG: Camera index out of range (expected: 'index < device_list->nb_devices'), where
>     'index' is 0
> must be less than
>     'device_list->nb_devices' is 0


[ERROR:0@90712.663] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


RuntimeError: 웹캠(0)을 열 수 없습니다. 번호(0/1) 확인.

In [12]:
import cv2

print("열리는 카메라 번호 탐색...")
found = []
for i in range(6):                       # 0~5번 시도
    cap = cv2.VideoCapture(i)
    if cap.isOpened():
        ret, frame = cap.read()          # 실제로 프레임이 읽히는지까지 확인
        if ret:
            print(f"  ✅ /dev/video{i} 열림 — 해상도 {frame.shape[1]}x{frame.shape[0]}")
            found.append(i)
        else:
            print(f"  ⚠ /dev/video{i} 열렸지만 프레임 못 읽음")
    cap.release()

if found:
    print(f"\n→ CAM_ID 를 {found[0]} 로 설정하세요.")
else:
    print("\n❌ 열리는 카메라 없음.")
    print("  USB캠: 케이블 재연결 후 'ls /dev/video*' 확인")
    print("  CSI캠(라즈베리캠 등): cv2.VideoCapture(0) 안 됨 → GStreamer 필요")

열리는 카메라 번호 탐색...
  ✅ /dev/video0 열림 — 해상도 640x480


[ WARN:0@90913.548] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index


  ✅ /dev/video1 열림 — 해상도 640x480

→ CAM_ID 를 0 로 설정하세요.


ioctl(VIDIOC_QBUF): 파일 디스크립터가 잘못됨
[ WARN:0@90913.920] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video2): can't open camera by index
[ WARN:0@90913.929] global cap.cpp:438 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(4.13.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1220: error: (-2:Unspecified error) in function 'bool CvCapture_FFMPEG::open(const char*, int, const cv::Ptr<cv::IStreamReader>&, const cv::VideoCaptureParameters&)'
> VIDEOIO/FFMPEG: Camera index out of range (expected: 'index < device_list->nb_devices'), where
>     'index' is 2
> must be less than
>     'device_list->nb_devices' is 2


[ERROR:0@90913.933] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range
[ WARN:0@90913.934] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video3): can't open camera by index
[ WARN:0@90913.935] global cap.cpp:438 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(4.13.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1220

In [13]:
import glob, cv2

# 실제 존재하는 /dev/video* 장치만 추려서 시도 (out of range 에러 방지)
devices = sorted(glob.glob("/dev/video*"))
print("인식된 장치:", devices)

found = []
for dev in devices:
    idx = int(dev.replace("/dev/video", ""))      # /dev/video0 → 0
    cap = cv2.VideoCapture(idx, cv2.CAP_V4L2)      # ★ V4L2 백엔드 명시(Nano 권장)
    if cap.isOpened():
        ret, frame = cap.read()
        if ret:
            print(f"  ✅ video{idx} 사용 가능 — {frame.shape[1]}x{frame.shape[0]}")
            found.append(idx)
        else:
            print(f"  ⚠ video{idx} 열렸으나 프레임 못 읽음")
    else:
        print(f"  ❌ video{idx} 열기 실패")
    cap.release()

if found:
    print(f"\n→ CAM_ID = {found[0]} 로 설정하세요.")
else:
    print("\n열리는 장치가 없습니다. 카메라 종류(USB/CSI)를 확인하세요.")

인식된 장치: ['/dev/video0', '/dev/video1']
  ✅ video0 사용 가능 — 640x480
  ❌ video1 열기 실패

→ CAM_ID = 0 로 설정하세요.


[ WARN:0@90955.259] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
[ WARN:0@90955.259] global cap.cpp:478 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by index


In [15]:
import os
import glob
import time
import subprocess
import cv2
import torch
from ultralytics import YOLO

# ══════════════════════════════════════════════════════════
# 0. 설정
# ══════════════════════════════════════════════════════════
CAM_ID = 0          # ★ 카메라 번호 0 고정
IMGSZ  = 320        # Nano 권장 해상도
CONF   = 0.5
SKIP   = 2          # 격프레임 추론

# ══════════════════════════════════════════════════════════
# 1. 환경 진단 (GPU torch 여부)
# ══════════════════════════════════════════════════════════
print("=" * 55)
print("torch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
GPU_OK = torch.cuda.is_available()
if not GPU_OK:
    print("⚠ CPU 전용 torch → CPU(ONNX)로 동작 (속도 제한적)")
print("=" * 55 + "\n")

DEVICE   = 0 if GPU_OK else "cpu"
USE_HALF = GPU_OK
EXPORT_FORMAT = "engine" if GPU_OK else "onnx"

# ══════════════════════════════════════════════════════════
# 2. Nano 성능 모드 (GPU일 때만, 실패 무시)
# ══════════════════════════════════════════════════════════
if GPU_OK:
    for cmd in (["sudo", "nvpmodel", "-m", "0"], ["sudo", "jetson_clocks"]):
        try:
            subprocess.run(cmd, check=True, timeout=10,
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception:
            pass

# ══════════════════════════════════════════════════════════
# 3. 가속 가중치 탐색 → 없으면 자동 변환
# ══════════════════════════════════════════════════════════
ext = "*.engine" if GPU_OK else "*.onnx"
accel = glob.glob(f"runs/**/weights/{ext}", recursive=True)
pt_files = glob.glob("runs/**/weights/best.pt", recursive=True) \
        or glob.glob("runs/**/weights/last.pt", recursive=True)

if accel:
    WEIGHTS = max(accel, key=os.path.getmtime)
    print("✅ 가속 모델 사용:", WEIGHTS)
elif pt_files:
    src_pt = max(pt_files, key=os.path.getmtime)
    print(f"⚙ 가속 모델 없음 → {EXPORT_FORMAT} 변환:", src_pt)
    YOLO(src_pt).export(
        format=EXPORT_FORMAT,
        imgsz=IMGSZ,
        half=USE_HALF,
        **({"device": 0} if GPU_OK else {}),
    )
    WEIGHTS = os.path.splitext(src_pt)[0] + (".engine" if GPU_OK else ".onnx")
    print("✅ 변환 완료:", WEIGHTS)
else:
    raise FileNotFoundError("runs 폴더에서 가중치(.pt)를 찾지 못했습니다.")

# ══════════════════════════════════════════════════════════
# 4. 모델 로드 + 클래스 확인
# ══════════════════════════════════════════════════════════
model = YOLO(WEIGHTS)
print("\n★ 모델 클래스:")
for idx, name in model.names.items():
    print(f"   {idx}: {name}")
print()

# ══════════════════════════════════════════════════════════
# 5. 웹캠 열기 (CAM_ID=0, V4L2 백엔드 명시)
# ══════════════════════════════════════════════════════════
cap = cv2.VideoCapture(CAM_ID, cv2.CAP_V4L2)   # ★ Nano USB캠은 V4L2가 안정적
if not cap.isOpened():
    cap = cv2.VideoCapture(CAM_ID)             # 실패 시 기본 백엔드로 재시도
if not cap.isOpened():
    raise RuntimeError(f"웹캠({CAM_ID})을 열 수 없습니다. 'ls /dev/video*'로 장치 확인.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)            # 밀린 프레임 누적 방지

print("실시간 탐지 시작 — 'q' 종료\n")
frame_count = 0
annotated = None
t_prev = time.time()

# ══════════════════════════════════════════════════════════
# 6. 추론 루프
# ══════════════════════════════════════════════════════════
while True:
    ret, frame = cap.read()
    if not ret:
        print("프레임 읽기 실패")
        break
    frame_count += 1

    if frame_count % SKIP == 0:
        results = model.predict(
            source=frame, imgsz=IMGSZ, conf=CONF,
            device=DEVICE, half=USE_HALF, verbose=False,
        )
        annotated = results[0].plot()
        for c, cf in zip(results[0].boxes.cls, results[0].boxes.conf):
            print(f"감지: {model.names[int(c)]} ({cf:.2f})")

    disp = annotated if annotated is not None else frame
    t_now = time.time()
    fps = 1.0 / max(t_now - t_prev, 1e-6)
    t_prev = t_now
    cv2.putText(disp, f"FPS: {fps:.1f}  {'GPU' if GPU_OK else 'CPU'}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow("WABS - Wheelchair Detection", disp)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
print("\n종료되었습니다.")

torch 버전: 2.12.0+cu130
CUDA 사용 가능: False
⚠ CPU 전용 torch → CPU(ONNX)로 동작 (속도 제한적)

✅ 가속 모델 사용: runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.onnx

★ 모델 클래스:
Loading runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 with CPUExecutionProvider
   0: empty_wheelchair
   1: person
   2: wheelchair_user
   3: people_wheelchair
   4: wheelchair
   5: wheelchairs
   6: wheelchairs_occ

실시간 탐지 시작 — 'q' 종료

Loading runs/detect/runs/wheelchair/yolov8n_4000_e50/weights/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 with CPUExecutionProvider
감지: empty_wheelchair (0.29)
감지: empty_wheelchair (0.59)
감지: empty_wheelchair (0.41)
감지: empty_wheelchair (0.52)
감지: empty_wheelchair (0.42)
감지: empty_wheelchair (0.32)
감지: empty_wheelchair (0.42)
감지: wheelchairs (0.61)
감지: people_wheelchair (0.28)
감지: wheelchairs (0.46)
감지: wheelchairs (0.50)
감지: wheelchairs (0.42)
감지: wheelchairs (0.39)
감지: wheelchairs (0.34)
감지: